## 다중 쿼리 검색기(MultiQueryRetriever)

- 거리 기반 벡터 데이터베이스 검색은 고차원 공간에서의 쿼리 임베딩(표현)과 '거리'를 기준으로 유사한 임베딩을 가진 문서를 찾는 방식
  
  **하지만 쿼리의 세부적인 차이나 임베딩이 데이터의 의미를 제대로 포착하지 못할 경우, 검색 결과가 달라짐**

  **또한, 이를 수동으로 조정하는 프롬프트 엔지니어링이나 튜닝 작업이 수반**

<br>

#### `MultiQueryRetriever`
-  주어진 사용자 입력 쿼리에 대해 다양한 관점에서 여러 쿼리를 자동으로 생성하는 LLM(Language Learning Model)을 활용해 프롬프트 튜닝 과정을 자동화
-  각각의 쿼리에 대해 관련 문서 집합을 검색하고, 모든 쿼리를 아우르는 고유한 문서들의 합집합을 추출해, 잠재적으로 관련된 더 큰 문서 집합을 얻을 수 있게 함
-  여러 관점에서 동일한 질문을 생성함으로써, `MultiQueryRetriever` 는 거리 기반 검색의 제한을 일정 부분 극복하고, 더욱 풍부한 검색 결과를 제공
  


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

<br>

#### 샘플 벡터 DB

In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
loader = WebBaseLoader(
    "https://teddylee777.github.io/openai/openai-assistant-tutorial/", encoding="utf-8"
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = loader.load_and_split(text_splitter)

In [6]:
openai_embedding = OpenAIEmbeddings()

db = FAISS.from_documents(docs, openai_embedding)

In [7]:
retriever = db.as_retriever()

query = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."
relevant_docs = retriever.invoke(query)

len(relevant_docs)

4

In [8]:
print(relevant_docs[1].page_content)

가장 강력한 도구로서, Assistant에게 사용자 정의 함수를 지정할 수 있습니다. 이는 Chat Completions API에서의 함수 호출과 매우 유사합니다.


Function calling(함수 호출) 도구를 사용하면 Assistant 에게 사용자 정의 함수 를 설명하여 호출해야 하는 함수를 인자와 함께 지능적으로 반환하도록 할 수 있습니다.


Assistant API는 실행 중에 함수를 호출할 때 실행을 일시 중지하며, 함수 호출 결과를 다시 제공하여 Run 실행을 계속할 수 있습니다. (이는 사용자 피드백을 받아 재게할 수 있는 의미이기도 합니다. 아래 튜토리얼에서 상세히 다룹니다).


<br>

### `MultiQueryRetriever`
- 사용할 LLM을 지정하고 질의 생성에 사용하면, retriever가 나머지 작업을 처리

In [9]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

In [10]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm,
)

In [11]:
question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."
relevant_docs = multiquery_retriever.invoke(question)

In [ ]:
print(
    f"===============\n검색된 문서 개수: {len(relevant_docs)}",
    end="\n===============\n",
)

print(relevant_docs[0].page_content)

검색된 문서 개수: 5
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용


<br>

### LCEL Chain 활용
- 질문을 입력 받으면 5개의 질문을 생성한 뒤 "\n" 구분자로 구분하여 생성된 5개 질문을 반환하는 Chain

In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

- 5개의 질문을 생성을 유도하는 프롬프트

In [15]:
prompt = PromptTemplate.from_template(
    """You are an AI language model assistant. 
    Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. 
    By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search. 
    Your response should be a list of values separated by new lines, eg: `foo\nbar\nbaz\n`

    #ORIGINAL QUESTION: 
    {question}

    #Answer in Korean:
    """
)

In [16]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

In [17]:
custom_multiquery_chain = (
    {"question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

In [18]:
question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."

multi_queries = custom_multiquery_chain.invoke(question)
multi_queries

'OpenAI Assistant API의 Functions 기능을 사용하는 방법을 설명해 주세요.  \nOpenAI Assistant API에서 Functions를 활용하는 방법에 대해 알고 싶습니다.  \nOpenAI Assistant API의 Functions를 어떻게 사용할 수 있는지 알려주세요.  \nOpenAI Assistant API의 Functions 사용법에 대한 자세한 정보를 제공해 주세요.  \nOpenAI Assistant API에서 Functions를 사용하는 절차를 설명해 주실 수 있나요?  '

- 이전에 생성한 Chain을 `MultiQueryRetriever` 에 전달하여 `retrieve` 

In [19]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    llm=custom_multiquery_chain, 
    retriever=db.as_retriever()
)

relevant_docs = multiquery_retriever.invoke(question)

print(
    f"===============\n검색된 문서 개수: {len(relevant_docs)}",
    end="\n===============\n",
)

print(relevant_docs[0].page_content)

검색된 문서 개수: 4
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용


<br>

<hr>

<br>

## 다중 벡터저장소 검색기(MultiVectorRetriever)
-  문서를 여러 벡터로 저장하고 관리할 수 있어, 정보 검색의 정확도와 효율성을 대폭 향상
  
<br>

### 문서당 여러 벡터 생성 방법
- **작은 청크 생성**: 문서를 더 작은 단위로 나눈 후, 각 청크에 대해 별도의 임베딩을 생성. 
- **요약 임베딩**: 각 문서의 요약을 생성하고, 이 요약으로부터 임베딩을 생성
  - 문서 전체를 분석하는 대신 핵심적인 요약 부분만을 활용하여 효율성을 극대화
- **가설 질문 활용**: 각 문서에 대해 적합한 가설 질문을 만들고, 이 질문에 기반한 임베딩을 생성
  - 특정 주제나 내용에 대해 깊이 있는 탐색을 원할 때 이 방법이 유용
  - 가설 질문은 문서의 내용을 다양한 관점에서 접근하게 해주며, 더 광범위한 이해를 가능하게 함
- **수동 추가 방식**: 사용자가 문서 검색 시 고려해야 할 특정 질문이나 쿼리를 직접 추가
  - 이 방법을 통해 사용자는 검색 과정에서 보다 세밀한 제어를 할 수 있으며, 자신의 요구 사항에 맞춘 맞춤형 검색이 가능

<br>

- 문서 로드

In [20]:
from langchain_community.document_loaders import PyMuPDFLoader

In [21]:
loader = PyMuPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

<br>

#### **Chunk + 원본 문서 검색**
- 대용량 정보를 검색하는 경우, 더 작은 단위로 정보를 임베딩하는 것이 유용
- `docstore` 에 원본 문서를 저장하고, `vectorstore` 에 임베딩된 문서를 저장
  
  $\rightarrow$ **문서를 더 작은 단위로 나누어 더 정확한 검색**

In [35]:
import uuid
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_vector import MultiVectorRetriever

In [36]:
vectorstore = Chroma(
    collection_name="small_bigger_chunks",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

# 부모 문서의 저장소
store = InMemoryStore()

In [37]:
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

In [38]:
doc_ids = [str(uuid.uuid4()) for _ in docs]
doc_ids

['41c97c5f-4d3e-443b-b95b-a8f6c6c59818',
 '71c8d405-60e3-414a-9020-9465f7457787',
 '09f927cd-5e96-4f20-b043-9295e6c55b97',
 'a881aab1-a3d2-4b63-8cb6-4b0c5ce2442e',
 'f5bd234b-cb05-480a-bc25-8962dd44bbae',
 '8d1943af-8a9a-4158-b51e-71210b9d025f',
 '4aef56b7-7904-45e3-95c7-e90a734dddfb',
 '880aabb0-fdeb-444c-b4f6-187d1794d046',
 '1d44aec2-8bc0-42f0-be22-ee1ce206b223',
 '5f4425bb-33e3-4143-8377-49238044228b',
 '04f53a67-98a9-49ab-b4ed-af71b2c45ec2',
 '76d36fd2-6ccb-4a92-a43e-4946d78d8b68',
 '052abd81-5e2c-4f53-8455-013b266d9ce2',
 '0a83582a-6c42-4eb4-9917-418b7ed3e2a3',
 'b7979d97-794b-4a59-8238-bb65ecafb0f7',
 'eb40b405-5247-405a-810b-270046dda229',
 '8aad9e9f-d5d1-4bf9-9e87-f7735896e5af',
 'c6886b7a-58a2-49ea-a0e8-1960d7a76d96',
 'efe70971-65c8-4f79-89d9-064c34c4b1b7',
 'bbd54b13-5896-456c-8133-87517307e635',
 '3d51fda3-31b1-435c-ae5b-fb586b58f57a',
 'dc80d465-41a8-4ebd-9e24-e0d66a494a0d',
 'fce81a2c-3ba6-4a6d-b5ad-c7c6b25c8c33']

- **큰 청크로 분할하기 위한 `parent_text_splitter`, 더 작은 청크로 분할하기 위한 `child_text_splitter`**

In [39]:
parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size=600)
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

- 더 큰 Chunk인 Parent 문서 생성

In [40]:
parent_docs = []

for i, doc in enumerate(docs):
    _id = doc_ids[i]
    
    # 현재 문서를 하위 문서로 분할
    parent_doc = parent_text_splitter.split_documents([doc])

    for _doc in parent_doc:
        # metadata에 문서 ID 를 저장
        _doc.metadata[id_key] = _id
    parent_docs.extend(parent_doc)
    
parent_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': 'data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0,
 'doc_id': '41c97c5f-4d3e-443b-b95b-a8f6c6c59818'}

- 더 작은 Chunk인 Child 문서 생성

In [41]:
child_docs = []
for i, doc in enumerate(docs):
    _id = doc_ids[i]
    
    # 현재 문서를 하위 문서로 분할
    child_doc = child_text_splitter.split_documents([doc])
    
    for _doc in child_doc:
        # metadata에 문서 ID 를 저장
        _doc.metadata[id_key] = _id
    child_docs.extend(child_doc)

child_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': 'data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0,
 'doc_id': '41c97c5f-4d3e-443b-b95b-a8f6c6c59818'}

In [42]:
print(f"분할된 parent_docs의 개수: {len(parent_docs)}")
print(f"분할된 child_docs의 개수: {len(child_docs)}")

분할된 parent_docs의 개수: 73
분할된 child_docs의 개수: 440


<br>

- 벡터저장소에 새롭게 생성한 작게 쪼개진 하위문서 집합을 추가
- 상위 문서는 생성한 UUID 와 맵핑하여 docstore 에 추가

In [ ]:
retriever.vectorstore.add_documents(parent_docs)
retriever.vectorstore.add_documents(child_docs)

retriever.docstore.mset(list(zip(doc_ids, docs)))

['f193f5b1-ffca-4460-a683-09ba4f0e5abf',
 'a3029155-896f-4f3c-b58b-d36f8f6aa8c0',
 '110c92b6-bf2f-4a69-a468-c0a6cf342aeb',
 '524bdc25-f02e-45d7-bfe7-65559ae573c5',
 'dfeb6cc4-5c0b-4273-9e5b-bbe7900ccea0',
 '6285e47a-abc9-4563-a75d-a3ad6a4cbc69',
 'd1e09cc7-5303-4581-a06d-bc67fbe73e74',
 '43bc77ff-55db-4bdb-9704-cd9c7a0cbfd3',
 '397267d4-c932-4064-aad5-c6949dfbc937',
 '17f13e0a-0362-462a-9731-38b55679bbea',
 'e6438796-0bee-4752-b2be-168b1e9eab85',
 '3a7533f3-bb6e-4e95-9b4f-feef210d4b61',
 '9d007125-6317-4437-a204-a3afd038cb60',
 '78295a44-fa59-4c92-97be-567a55bb3c12',
 '0df0066b-bd88-4aeb-87c7-810e0b91a317',
 '22f91e8a-b687-47f3-a9c8-0c3395b3d01c',
 'f52095dc-0311-43bf-9645-f4e855b0e5a6',
 'ebe3c33c-39b8-495e-9090-a350e7ce7ebd',
 '65642bde-2990-4a56-ae31-f243a82df063',
 '8152eba2-159e-401f-b241-c94e9241b9a9',
 '035c9158-eff5-4832-ad6f-84909b67cf6f',
 'fad9593e-a137-4e4e-ace6-750a92fd171f',
 'fc426e22-c059-4ef2-9ecc-df9a955465a2',
 '7c8e07e5-541b-48c6-a34c-6589e9f3bd50',
 '79f654a3-c2ce-